In [0]:
%sql
select * from de_project.bronze_layer.crm_cust_info;

1- wrong column names
2- column issues

Steps to take
1- change the column names
2- Set the column name accordingly
3- Remove the null and duplicate values
4- Rename M into Married and S into Single
5- Rename F into femlae and M into Male
6- Validiate String Variable
7-

# Read the Data from Bronze Layer

In [0]:
df = spark.table('de_project.bronze_layer.crm_cust_info')

In [0]:
df.display()

# Data Cleaning


## Finding and removing Duplicates

In [0]:
#Here I will try to find the duplicates in the Dataframe most important columns such as Customer_id and Customer_key (Just to make sure the names of the coloumns are not changed yet use the previous names )
df.groupBy('cst_id','cst_key').count().filter('count>1').show()

In [0]:
#Here I will be dropping the duplicate values from the that both columns
df = df.dropDuplicates(['cst_id','cst_key'])

In [0]:
#Dropped the duplicate values just cheking them
df.groupBy('cst_id','cst_key').count().filter('count>1').show()

## Finding and Removing Null Values

In [0]:
# Here I will be finding and dropping the Null values
df.filter(df.cst_id.isNull()).show()

In [0]:
df= df.dropna(subset=['cst_id','cst_key'])

# Data Transformation

- 1 So first I will standrise the data

In [0]:
df.schema


## Trimming

In [0]:
#here im going to trim the columns names means the spaces in the start and at the end of the cloumns will be removed
# And going to make the cloumns lower cases
from pyspark.sql.functions import trim,col,lower

from pyspark.sql.types import StringType

In [0]:
for feild in df.schema.fields:
  if isinstance(feild.dataType, StringType):
      df = df.withColumn(feild.name,lower(trim(col(feild.name))))

In [0]:
df.display()

## Normalization

In [0]:
import pyspark.sql.functions as F
df = df.withColumn(
    "cst_marital_status",
    F.when(
        F.upper(F.col("cst_marital_status")) == "S", "Single"
    ).when(
        F.upper(F.col("cst_marital_status")) == "M", "Married"
    ).otherwise("n/a")
)
df = df.withColumn(
    "cst_gndr",
    F.when(
        F.upper(F.col("cst_gndr")) == 'M' , 'Male')
    .when(
        F.upper(F.col('cst_gndr')) =='F' , 'Female')
    .otherwise('n/a')
)


# Renaming if Columns
 

In [0]:
df.display()

In [0]:
# here im going to make the distorny and then going to store all the cloumns in that dictonery
rename_dict = {
    'cst_id': 'Customer_id',
    'cst_key' : 'Customer_key',
    'cst_firstname' : 'Customer_firstname',
    'cst_lastname' : 'Customer_lastname',
    'cst_marital_status' : 'Customer_marital_status',
    'cst_gndr' : 'Customer_gender' ,
    'cst_create_date' : 'Customer_create_date'
    }

In [0]:
print(rename_dict)

In [0]:
for old , new in rename_dict.items():
    df = df.withColumnRenamed(old,new)

In [0]:
df.display(5)

# Write into the Silver Tabel

In [0]:
(
    df.write.mode('overwrite').format('delta').saveAsTable('de_project.silver_layer.crm_customer')
)

In [0]:
%sql
select * from de_project.silver_layer.crm_customer;

In [0]:
%sql
select * from de_project.silver_layer.crm_customer
where Customer_id is null;

In [0]:
%sql 
select * from de_project.silver_layer.crm_customer
where Customer_key is null;

In [0]:
%sql 
select * from de_project.silver_layer.crm_customer
where Customer_firstname is null;

In [0]:
%sql 
select * from de_project.silver_layer.crm_customer
where Customer_lastname is null;

In [0]:
df.groupBy('Customer_id').count().filter('count>1').show()

In [0]:
%sql 
select Customer_id, count(*) from de_project.silver_layer.crm_customer
groupby Customer_id
having count(*)>1;